# Maintenance

In [ ]:
import itertools
import os

from sqlalchemy import delete, func, select
from assistant_agent.entities.checkpoint import (
    CheckpointBlobEntity,
    CheckpointEntity,
    CheckpointWriteEntity,
)
from assistant_agent.store import PostgresStoreConnector

engine = PostgresStoreConnector(os.environ["AA_PG_CONNECTION_STRING"]).get_engine()

## Checkpointer
旧 thread_id（`agent_id` ごとの世代交代で使われなくなった checkpoint）を手動で削除する。

### 削除対象を表示（確認用）

In [ ]:
KEEP_LATEST_N = 0  # agent_id ごとに残す世代数

agent_id_col = func.split_part(CheckpointEntity.thread_id, ":", 1)
async with engine.connect() as conn:
    stmt = (
        select(
            agent_id_col.label("agent_id"),
            CheckpointEntity.thread_id,
            func.max(CheckpointEntity.checkpoint_id).label("latest"),
        )
        .group_by(agent_id_col, CheckpointEntity.thread_id)
        .order_by(agent_id_col, func.max(CheckpointEntity.checkpoint_id).desc())
    )
    rows = (await conn.execute(stmt)).all()

stale_thread_ids = []
for agent_id, group in itertools.groupby(rows, key=lambda row: row.agent_id):
    stale_thread_ids += [row.thread_id for row in list(group)[KEEP_LATEST_N:]]
print(f"Deleting {len(stale_thread_ids)} thread(s):\n{'\n'.join(stale_thread_ids)}")

### 実際に削除する（セル1を確認してから実行）

In [ ]:
async with engine.begin() as conn:
    for entity in (CheckpointEntity, CheckpointBlobEntity, CheckpointWriteEntity):
        await conn.execute(delete(entity).where(entity.thread_id.in_(stale_thread_ids)))

# Dispatcher

### レコードを表示

In [ ]:
from sqlalchemy import select
from assistant_agent.entities.postgres import DispatchEntity

async with engine.connect() as conn:
    stmt = select(DispatchEntity).order_by(DispatchEntity.run_at)
    dispatch_rows = (await conn.execute(stmt)).all()

for row in dispatch_rows:
    print(row)

### 予定を追加

In [ ]:
import uuid
from datetime import UTC, datetime, timedelta
from sqlalchemy import insert
from assistant_agent.entities.postgres import DispatchEntity

ONE_SHOT = -1  # DispatcherService.ONE_SHOT と同じ値（単発予定）

new_dispatch = {
    "dispatch_id": str(uuid.uuid7()),
    "prompt": "メンテナンス確認用の予定",
    "interval_seconds": ONE_SHOT,
    "next_fire_at": datetime.now(UTC) + timedelta(seconds=10),
}
async with engine.begin() as conn:
    await conn.execute(insert(DispatchEntity), [new_dispatch])
print(new_dispatch)

### 予定を更新

In [ ]:
from sqlalchemy import update
from assistant_agent.entities.postgres import DispatchEntity

update_dispatch_id = ""  # 更新したい dispatch_id を指定（レコード表示セルの結果から確認）
update_prompt = None  # 変更する場合のみ文字列を指定（None なら変更しない）
update_interval_seconds = None  # 変更する場合のみ int を指定（None なら変更しない）
update_next_fire_at = None  # 変更する場合のみ tz-aware datetime を指定（None なら変更しない）

values = {
    k: v
    for k, v in {
        "prompt": update_prompt,
        "interval_seconds": update_interval_seconds,
        "next_fire_at": update_next_fire_at,
    }.items()
    if v is not None
}
async with engine.begin() as conn:
    result = await conn.execute(
        update(DispatchEntity).where(DispatchEntity.dispatch_id == update_dispatch_id).values(**values)
    )
print(f"Updated {result.rowcount} row(s): {update_dispatch_id} -> {values}")

### 予定を削除

In [ ]:
from sqlalchemy import delete
from assistant_agent.entities.postgres import DispatchEntity

delete_dispatch_id = ""  # 削除したい dispatch_id を指定（レコード表示セルの結果から確認）

async with engine.begin() as conn:
    result = await conn.execute(
        delete(DispatchEntity).where(DispatchEntity.dispatch_id == delete_dispatch_id)
    )
print(f"Deleted {result.rowcount} row(s): {delete_dispatch_id}")